# SSI Neuro-Prosthetic — Calibration Training Pipeline

**Notebook:** `01_calibration_training.ipynb`  
**Purpose:** Train a 1D-CNN regression model to map 3-channel subvocal biosignal windows  
to a continuous 6-DOF vocal expression vector.

---

## Architecture Summary

| Stage | Layer | Output Shape | Notes |
|---|---|---|---|
| Input | — | `(512, 3)` | 512 samples × 3 channels (L-sEMG, R-sEMG, MMG) |
| Block 1 | Conv1D(64, 7) → BN → ReLU → MaxPool(2) | `(256, 64)` | Captures low-freq phoneme envelope |
| Block 2 | Conv1D(128, 5) → BN → ReLU → MaxPool(2) | `(128, 128)` | Mid-freq formant dynamics |
| Block 3 | Conv1D(256, 3) → BN → ReLU → MaxPool(2) | `(64, 256)` | High-freq transient features |
| Pooling | GlobalAveragePooling1D | `(256,)` | Spatial invariance, reduces overfitting |
| FC 1 | Dense(128) → ReLU → Dropout(0.3) | `(128,)` | — |
| Output | Dense(6, activation=**linear**) | `(6,)` | 6-DOF: pitch, yaw, intensity, F1, F2, F3 |

**Loss:** Mean Squared Error (MSE) — continuous regression  
**Optimizer:** Adam (lr=1e-3)  
**Backend:** `tensorflow-macos` + `tensorflow-metal` (Metal GPU, Apple M-series)


In [ ]:
# ── Environment & Backend Verification ──────────────────────────────────────
import platform
import numpy as np
import tensorflow as tf

print(f"Python   : {platform.python_version()}")
print(f"Platform : {platform.machine()} — {platform.system()}")
print(f"NumPy    : {np.__version__}")
print(f"TF       : {tf.__version__}")

# Verify Metal GPU is visible
gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPU devices: {gpus}")
if gpus:
    print("✅ Metal GPU acceleration active")
else:
    print("⚠️  No Metal GPU found — running on CPU")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import struct
import glob

import numpy as np
import scipy.signal as sig
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Reproducibility seed
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Imports OK")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Signal parameters (must match firmware constants)
SAMPLE_RATE    = 2000       # Hz
WINDOW_SIZE    = 512        # Samples per frame
N_CHANNELS     = 3          # L-sEMG, R-sEMG, MMG

# Output dimensions
N_OUTPUT_DIMS  = 6          # 6-DOF: pitch, yaw, intensity, F1, F2, F3
OUTPUT_LABELS  = ['pitch', 'yaw', 'intensity', 'formant_1', 'formant_2', 'formant_3']

# Training hyperparameters
BATCH_SIZE     = 32
EPOCHS         = 100
LEARNING_RATE  = 1e-3
DROPOUT_RATE   = 0.3
VAL_SPLIT      = 0.2
TEST_SPLIT     = 0.1

# Binary frame constants (must match firmware CalibrationFrameHeader)
MAGIC          = b'SSI0'
HEADER_FMT     = '<4sIIHH'   # magic[4], timestamp_ms, sample_count, num_ch, samples_per_ch
HEADER_SIZE    = struct.calcsize(HEADER_FMT)

print(f"Window size   : {WINDOW_SIZE} samples @ {SAMPLE_RATE} Hz = {WINDOW_SIZE/SAMPLE_RATE*1000:.1f} ms")
print(f"Output dims   : {N_OUTPUT_DIMS} ({', '.join(OUTPUT_LABELS)})")
print(f"Header size   : {HEADER_SIZE} bytes")

In [ ]:
# ── SD Card Binary Parser ─────────────────────────────────────────────────────

def parse_ssi_binary(filepath: str) -> list[dict]:
    """
    Parse a .ssi calibration file produced by the Teensy firmware.

    Binary layout per frame:
      [Header: 16 bytes]
        magic[4]        = b'SSI0'
        timestamp_ms    = uint32 (little-endian)
        sample_count    = uint32
        num_channels    = uint16
        samples_per_ch  = uint16
      [Data: num_channels * samples_per_ch * 2 bytes]
        uint16[num_channels][samples_per_ch]  (channel-major order)

    Returns:
      List of dicts with keys:
        'timestamp_ms' : int
        'sample_count' : int
        'data'         : np.ndarray shape (num_channels, samples_per_ch), dtype float32
    """
    frames = []
    with open(filepath, 'rb') as f:
        while True:
            hdr_bytes = f.read(HEADER_SIZE)
            if len(hdr_bytes) < HEADER_SIZE:
                break

            magic, ts_ms, samp_cnt, n_ch, n_samp = struct.unpack(HEADER_FMT, hdr_bytes)

            # Validate magic
            if magic != MAGIC:
                print(f"  ⚠️  Bad magic at byte offset — skipping frame")
                break

            # Read raw data block
            n_samples_total = int(n_ch) * int(n_samp)
            raw_bytes = f.read(n_samples_total * 2)  # uint16 = 2 bytes each
            if len(raw_bytes) < n_samples_total * 2:
                print("  ⚠️  Truncated frame — stopping")
                break

            # Decode as uint16, reshape to [channels, samples], convert to µV
            raw_u16 = np.frombuffer(raw_bytes, dtype='<u2').reshape(n_ch, n_samp)
            # Convert: (ADC_val - 2048) * (3.3V / 4096) * 1e6 µV
            data_uv = (raw_u16.astype(np.float32) - 2048.0) * 805.7

            frames.append({
                'timestamp_ms': ts_ms,
                'sample_count': samp_cnt,
                'data': data_uv  # shape (3, 512)
            })

    return frames


def load_calibration_dataset(data_dir: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Load all .ssi files from data_dir.
    Labels are expected in a sidecar .npy file: <filename>.labels.npy
    Each labels file shape: (n_frames, 6) — the 6-DOF expression vectors
    annotated by the clinician during calibration.

    Returns:
        X: np.ndarray shape (N, WINDOW_SIZE, N_CHANNELS)
        y: np.ndarray shape (N, N_OUTPUT_DIMS)
    """
    X_list, y_list = [], []

    for ssi_path in sorted(glob.glob(os.path.join(data_dir, '*.ssi'))):
        label_path = ssi_path + '.labels.npy'
        if not os.path.exists(label_path):
            print(f"  Skipping {os.path.basename(ssi_path)} — no label file")
            continue

        frames = parse_ssi_binary(ssi_path)
        labels = np.load(label_path)  # shape: (n_frames, 6)

        if len(frames) != len(labels):
            print(f"  ⚠️  Frame/label mismatch in {os.path.basename(ssi_path)}")
            continue

        for frame, label in zip(frames, labels):
            # Transpose data to (WINDOW_SIZE, N_CHANNELS) for Conv1D
            X_list.append(frame['data'].T)   # (512, 3)
            y_list.append(label)              # (6,)

    if not X_list:
        raise FileNotFoundError(f"No labeled .ssi files found in: {data_dir}")

    return np.stack(X_list).astype(np.float32), np.stack(y_list).astype(np.float32)


print("Binary parser defined. Run load_calibration_dataset('./data') when .ssi files are available.")

In [ ]:
# ── Synthetic Dataset Generator (for architecture validation without hardware) ─

def generate_synthetic_dataset(n_samples: int = 500, seed: int = SEED) -> tuple[np.ndarray, np.ndarray]:
    """
    Generates a synthetic calibration dataset for model architecture validation.

    X: Bandpass-filtered Gaussian noise (20-400 Hz sEMG-like signals)
       Shape: (n_samples, WINDOW_SIZE, N_CHANNELS)
    y: Random 6-DOF expression vectors in [-1, 1] (except intensity: [0, 1])
       Shape: (n_samples, N_OUTPUT_DIMS)

    Frequency bands:
      CH0 (L-sEMG): 20–200 Hz bandpass
      CH1 (R-sEMG): 20–200 Hz bandpass
      CH2 (MMG-PZT): 5–80 Hz bandpass (tissue vibration, lower freq)
    """
    rng = np.random.default_rng(seed)

    # Bandpass filter specs
    nyq = SAMPLE_RATE / 2.0
    bands = [
        (20.0 / nyq, 200.0 / nyq),   # L-sEMG
        (20.0 / nyq, 200.0 / nyq),   # R-sEMG
        ( 5.0 / nyq,  80.0 / nyq),   # MMG PZT
    ]

    X = np.zeros((n_samples, WINDOW_SIZE, N_CHANNELS), dtype=np.float32)
    for ch, (lo, hi) in enumerate(bands):
        b, a = sig.butter(4, [lo, hi], btype='bandpass')
        noise = rng.standard_normal((n_samples, WINDOW_SIZE))
        for i in range(n_samples):
            X[i, :, ch] = sig.filtfilt(b, a, noise[i]).astype(np.float32)

    # Normalize each channel independently
    for ch in range(N_CHANNELS):
        std = X[:, :, ch].std() + 1e-8
        X[:, :, ch] /= std

    # Labels: [-1, 1] for most dims, [0, 1] for intensity
    y = rng.uniform(-1.0, 1.0, (n_samples, N_OUTPUT_DIMS)).astype(np.float32)
    y[:, 2] = np.abs(y[:, 2])  # intensity = [0, 1]

    return X, y


print("Generating synthetic dataset for architecture validation...")
X_synth, y_synth = generate_synthetic_dataset(n_samples=500)
print(f"X shape: {X_synth.shape}  (samples, window, channels)")
print(f"y shape: {y_synth.shape}  (samples, output_dims)")

In [ ]:
# ── Data Visualization ────────────────────────────────────────────────────────

fig = plt.figure(figsize=(14, 8))
fig.suptitle('SSI Calibration Frame — Synthetic Preview', fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.3)

channel_names = ['L-sEMG', 'R-sEMG', 'MMG (PZT)']
colors = ['#2196F3', '#4CAF50', '#FF9800']
t = np.arange(WINDOW_SIZE) / SAMPLE_RATE * 1000  # ms

# Time-domain plots
for ch in range(N_CHANNELS):
    ax = fig.add_subplot(gs[0, ch])
    ax.plot(t, X_synth[0, :, ch], color=colors[ch], lw=0.8, alpha=0.9)
    ax.set_title(channel_names[ch], fontsize=11)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Amplitude (norm.)')
    ax.grid(True, alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

# Frequency spectra
for ch in range(N_CHANNELS):
    ax = fig.add_subplot(gs[1, ch])
    freqs = np.fft.rfftfreq(WINDOW_SIZE, 1.0 / SAMPLE_RATE)
    psd = np.abs(np.fft.rfft(X_synth[0, :, ch])) ** 2
    ax.semilogy(freqs, psd, color=colors[ch], lw=0.9, alpha=0.9)
    ax.set_title(f'{channel_names[ch]} — PSD', fontsize=11)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Power')
    ax.set_xlim(0, SAMPLE_RATE // 2)
    ax.grid(True, alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('../figures/synthetic_signal_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/synthetic_signal_preview.png")

In [ ]:
# ── 1D-CNN Model Definition ───────────────────────────────────────────────────

def build_ssi_cnn(
    input_shape: tuple = (WINDOW_SIZE, N_CHANNELS),
    output_dims: int   = N_OUTPUT_DIMS,
    dropout_rate: float = DROPOUT_RATE
) -> keras.Model:
    """
    Build the SSI 1D-CNN regression model.

    Architecture:
      Input → [Conv1D → BatchNorm → ReLU → MaxPool] × 3
             → GlobalAveragePooling1D
             → Dense(128, ReLU) → Dropout
             → Dense(6, linear)   ← regression output, no activation

    The LINEAR output layer is critical: this is a multi-dimensional
    continuous regression task, NOT discrete classification.
    Output range is unconstrained — normalization is applied during
    training via label scaling.

    Args:
        input_shape  : (WINDOW_SIZE, N_CHANNELS) = (512, 3)
        output_dims  : Number of regression targets = 6
        dropout_rate : Dropout probability (applied after Dense(128))

    Returns:
        Uncompiled Keras Model
    """
    inputs = keras.Input(shape=input_shape, name='biosignal_input')

    # ── Convolutional Block 1: Low-frequency phoneme envelope (kernel=7) ──
    x = layers.Conv1D(64, kernel_size=7, padding='same', use_bias=False, name='conv1')(inputs)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.ReLU(name='relu1')(x)
    x = layers.MaxPooling1D(pool_size=2, name='pool1')(x)   # → (256, 64)

    # ── Convolutional Block 2: Mid-frequency formant dynamics (kernel=5) ──
    x = layers.Conv1D(128, kernel_size=5, padding='same', use_bias=False, name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.ReLU(name='relu2')(x)
    x = layers.MaxPooling1D(pool_size=2, name='pool2')(x)   # → (128, 128)

    # ── Convolutional Block 3: High-frequency transients (kernel=3) ────────
    x = layers.Conv1D(256, kernel_size=3, padding='same', use_bias=False, name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)
    x = layers.ReLU(name='relu3')(x)
    x = layers.MaxPooling1D(pool_size=2, name='pool3')(x)   # → (64, 256)

    # ── Global Average Pooling → compact feature vector ───────────────────
    # Provides spatial invariance and reduces overfitting vs. Flatten
    x = layers.GlobalAveragePooling1D(name='gap')(x)         # → (256,)

    # ── Fully Connected Regression Head ───────────────────────────────────
    x = layers.Dense(128, activation='relu', name='fc1')(x)
    x = layers.Dropout(dropout_rate, name='dropout')(x)

    # ── Linear Output: 6-DOF Continuous Expression Vector ─────────────────
    # activation=None (default) = linear — required for regression
    # Output dimensions: [pitch, yaw, intensity, formant_1, formant_2, formant_3]
    outputs = layers.Dense(output_dims, activation=None, name='expression_output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='SSI_1D_CNN')
    return model


model = build_ssi_cnn()
model.summary()

In [ ]:
# ── Data Preprocessing & Splitting ────────────────────────────────────────────

# Use synthetic data; swap for load_calibration_dataset('./data') when hardware is available
X, y = X_synth, y_synth

# Train/Val/Test split: 70 / 20 / 10
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SPLIT, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=VAL_SPLIT / (1.0 - TEST_SPLIT),
    random_state=SEED
)

# Label normalization: StandardScaler on training labels only
# (prevents data leakage from val/test into training distribution)
label_scaler = StandardScaler()
y_train_scaled = label_scaler.fit_transform(y_train)
y_val_scaled   = label_scaler.transform(y_val)
y_test_scaled  = label_scaler.transform(y_test)

print(f"Train   : {X_train.shape[0]} samples")
print(f"Val     : {X_val.shape[0]} samples")
print(f"Test    : {X_test.shape[0]} samples")
print(f"X shape : {X_train.shape}  (samples, time, channels)")
print(f"y shape : {y_train_scaled.shape}  (samples, output_dims)")

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='mse',
    metrics=['mae']
)

training_callbacks = [
    # Early stopping: stop if val_loss doesn't improve for 15 epochs
    callbacks.EarlyStopping(
        monitor='val_loss', patience=15,
        restore_best_weights=True, verbose=1
    ),
    # Reduce LR on plateau: halve lr after 8 epochs no improvement
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=8, min_lr=1e-6, verbose=1
    ),
    # Model checkpoint: save best weights
    callbacks.ModelCheckpoint(
        filepath='../models/ssi_cnn_best.keras',
        monitor='val_loss', save_best_only=True, verbose=0
    ),
    # TensorBoard logging
    callbacks.TensorBoard(log_dir='../logs/tensorboard', histogram_freq=1),
]

os.makedirs('../models', exist_ok=True)
os.makedirs('../logs/tensorboard', exist_ok=True)

print(f"Training for up to {EPOCHS} epochs (batch={BATCH_SIZE}, lr={LEARNING_RATE})...")
history = model.fit(
    X_train, y_train_scaled,
    validation_data=(X_val, y_val_scaled),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=training_callbacks,
    verbose=1
)

In [ ]:
# ── Training Curves ───────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('SSI 1D-CNN — Training History', fontsize=13, fontweight='bold')

axes[0].plot(history.history['loss'],     label='Train MSE', color='#2196F3')
axes[0].plot(history.history['val_loss'], label='Val MSE',   color='#FF5722', linestyle='--')
axes[0].set_title('Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].plot(history.history['mae'],     label='Train MAE', color='#4CAF50')
axes[1].plot(history.history['val_mae'], label='Val MAE',   color='#FF9800', linestyle='--')
axes[1].set_title('MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
os.makedirs('../figures', exist_ok=True)
plt.savefig('../figures/training_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Evaluation on Test Set ────────────────────────────────────────────────────

test_loss, test_mae = model.evaluate(X_test, y_test_scaled, verbose=0)
print(f"Test MSE : {test_loss:.6f}")
print(f"Test MAE : {test_mae:.6f}")

# Per-dimension MAE (unscaled, in original label units)
y_pred_scaled = model.predict(X_test, verbose=0)
y_pred = label_scaler.inverse_transform(y_pred_scaled)
y_true = label_scaler.inverse_transform(y_test_scaled)

per_dim_mae = np.mean(np.abs(y_pred - y_true), axis=0)
print("\nPer-dimension MAE (original scale):")
for label, mae in zip(OUTPUT_LABELS, per_dim_mae):
    print(f"  {label:12s}: {mae:.4f}")

In [ ]:
# ── Export Model for TFLite-Micro Deployment ──────────────────────────────────

# Step 1: Save full Keras model
model.save('../models/ssi_cnn_final.keras')
print("Saved: ../models/ssi_cnn_final.keras")

# Step 2: Convert to TFLite flatbuffer (float32)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = []
tflite_model = converter.convert()

tflite_path = '../models/ssi_cnn.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f"Saved: {tflite_path} ({len(tflite_model)/1024:.1f} KB)")

# Step 3: Convert flatbuffer to C array for Teensy PROGMEM inclusion
hex_values = ', '.join(f'0x{b:02x}' for b in tflite_model)
c_header = f"""// Auto-generated by 01_calibration_training.ipynb
// DO NOT EDIT MANUALLY
#pragma once
#include <stdint.h>
static const uint8_t SSI_MODEL_DATA[] = {{{hex_values}}};
static const uint32_t SSI_MODEL_SIZE = {len(tflite_model)};
"""

os.makedirs('../../firmware-teensy/include', exist_ok=True)
with open('../../firmware-teensy/include/ssi_model_data.h', 'w') as f:
    f.write(c_header)
print("Saved: ../../firmware-teensy/include/ssi_model_data.h")
print(f"\n✅ TFLite model ready for Teensy deployment ({len(tflite_model)} bytes)")

## Next Steps

1. **Collect real calibration data** — Flash firmware to Teensy 4.1, attach sEMG electrodes to submental region, record `.ssi` files via SD card
2. **Annotate labels** — Use a clinical annotation tool to generate `.labels.npy` sidecar files (6-DOF expression vectors per 512-sample window)
3. **Retrain on real data** — Replace `X_synth, y_synth` with `load_calibration_dataset('./data')` in the training cell
4. **Deploy to Teensy** — The exported `ssi_model_data.h` embeds the TFLite flatbuffer into firmware flash for inference via `lib/tflite-micro`
5. **Quantize for performance** — Enable `converter.optimizations = [tf.lite.Optimize.DEFAULT]` for INT8 quantization to reduce model size and improve inference speed on M7
